# GP02C Live Attachment

Inspect the early organizer-worker attachment for the revision controller.

This source copy comes from **GP02C_04B_Live_ADK_Gemma_Integration_v1_1.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

# GP02C-04B v1.1 — LIVE Google ADK / Gemma 4 31B Worker Integration

**Research:** *When Passing Tests Become Stale: Evidence Freshness in Gemma 4 Software Repair Agents*  
**Experimental checkpoint:** The actual L4 × 4 / vLLM / Google ADK worker, not just CPU integration preparation.  
**Task:** `rich_2943` only. **Arms:** B (final-state gate) and C (continuous freshness).  
**Baseline:** Exact source recovered from the user's `GP02B_v2_7_rich_2943.zip` historical run evidence. **No changes** to the frozen GP02B historical notebook or results.

### Execution order

1. Read and pin the real archived worker source and approved controller modules (CPU).
2. Restore the original GP02B package inputs and official offline wheel environment (CPU).
3. Load the original Gemma 4 31B IT QAT checkpoint on **L4 × 4** using the original vLLM settings.
4. Build a **separate** experimental derivative, attaching GP02C after the existing ADK tool-policy wrapper.
5. Qualify the development task using the existing unchanged GP02B procedure; do not pass evaluation-only material to Gemma.
6. Run the actual Gemma worker once for B and then C, subject to validated live readiness and available quota.
7. Archive a uniquely named ZIP per arm, including original worker traces and immutable GP02C evidence; run independent verification *after* the repair, without feedback to the agent.

**Research boundary:** A successful CPU check does not mean live integration passed. The notebook must produce real `ACTUAL_ADK_REGISTRY_ACCEPTED` and model-use records before claiming a live test. If compatibility fails, record the hold and stop rather than bypass it. Do not run the 36 held-out evaluation candidates.

**Runtime requirement:** Attach the official competition inputs, original 31B QAT checkpoint, offline wheelhouse, one historical `GP02B_v2_7_rich_2943.zip`, and `GP02C_04_CPU_Evidence_v1_0.zip` (or original approved GP02C-04 package containing identical pinned source modules).

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, zipfile, os, sys, shutil, subprocess, uuid, importlib.util, copy, dataclasses, time
from IPython.display import display, FileLink

RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
SESSION_ROOT = Path('/kaggle/working') / ('GP02C_04B_Live_'+RUN_TAG)
SESSION_ROOT.mkdir(parents=True, exist_ok=False)
SOURCE_DIR = SESSION_ROOT/'approved_source'
SOURCE_DIR.mkdir()
EVIDENCE_ROOT = SESSION_ROOT/'attempt_evidence'
EVIDENCE_ROOT.mkdir()
PILOT_ROOT = SESSION_ROOT/'agent_trials'
PILOT_ROOT.mkdir()
TASK_ID='rich_2943'
SEED=42
ARMS=('B','C')
FROZEN_WORKER_SHA='dbbdf4682dccb6d1b44a4d4dca3bb90a89b5b9bbe9070d4c36c16ec31e3eebfb'
PINNED_CONTROLLER_SHA256 = {'gp02c_ledger.py': '49b62a61e8ca930706acb198b13cdd89d4f563ad4f884a98dc1292ea13757c76', 'gp02c_live_adapter.py': 'b470571de6f59d7862573773ca8a08d389369f4eb1312cbd70aa8575ff72e7c5', 'gp02c_adk_bridge.py': '49431d2f9482b2d7bcfb53a0f6a3a6d3ac4322dd542bf6b37a169f88d4b6a754', 'gp02c04_pilot_wiring.py': 'cdeb2b8007aedb355593d15ea2032904dac01201f0a684abf6e201860d00ac9a', 'gp02c_worker_supervisor.py': 'df9808f7d08032138c9b91a22302084f691052503f4eee16a67bc9cfe28714f8', 'gp02c_evidence_export.py': '2c014fc28300d7a54eabace794ed49f57403750191a85ed1fbdaec10f602d768'}
print('LIVE GP02C-04B isolated workspace:', SESSION_ROOT)
print('Configured tasks:',TASK_ID,'| arms:',ARMS,'| baseline source unchanged')

## 1. Authentic frozen-source acquisition (CPU)

The GP02B development archives already contain the **real** version 2.7 worker, prompt components, code, configuration and starter submission. We read the existing evidence instead of trying to reproduce the frozen worker by applying unreviewed changes. The loaded GP02C-04 modules are required to match their recorded hashes exactly. No hidden evaluator data are copied to agent-facing paths.

In [ ]:
SEARCH_ROOTS=(Path('/kaggle/input'),Path('/kaggle/working'))
def find_archives(needle):
    matches=[]
    for root in SEARCH_ROOTS:
        if root.is_dir():
            matches += [p for p in root.rglob('*.zip') if needle.lower() in p.name.lower()]
    return sorted(set(matches))

BASELINE_ARCHIVES=find_archives('GP02B_v2_7_rich_2943')
CONTROL_ARCHIVES=find_archives('GP02C_04_CPU_Evidence')+find_archives('GP02C_04_ADK_Integration_and_Evidence_CPU_Package')
if not BASELINE_ARCHIVES or not CONTROL_ARCHIVES:
    raise FileNotFoundError('Attach the historical GP02B_v2_7_rich_2943 evidence ZIP and GP02C-04 approved controller source ZIP to this Kaggle notebook.')

BASELINE_ZIP=None
for candidate in BASELINE_ARCHIVES:
    with zipfile.ZipFile(candidate) as z:
        if 'run_sources/smoke_worker.py' in z.namelist() and hashlib.sha256(z.read('run_sources/smoke_worker.py')).hexdigest()==FROZEN_WORKER_SHA:
            BASELINE_ZIP=candidate;break
if BASELINE_ZIP is None:
    raise RuntimeError('None of the attached GP02B sources matches the frozen v2.7 worker SHA')

CONTROLLER_ZIP=None
APPROVED_SOURCE={}
for candidate in CONTROL_ARCHIVES:
    with zipfile.ZipFile(candidate) as z:
        seen={}
        for path in z.namelist():
            name=Path(path).name
            if name in PINNED_CONTROLLER_SHA256:
                if name in seen: raise RuntimeError('Duplicate controller module in candidate archive: '+name)
                data=z.read(path)
                if hashlib.sha256(data).hexdigest()!=PINNED_CONTROLLER_SHA256[name]:
                    seen={}; break
                seen[name]=data
        if set(seen)==set(PINNED_CONTROLLER_SHA256):
            CONTROLLER_ZIP=candidate
            APPROVED_SOURCE=seen
            break
if CONTROLLER_ZIP is None:
    raise RuntimeError('Approved controller byte identities do not match the attached GP02C-04 archive')

for name,data in APPROVED_SOURCE.items():
    compile(data.decode('utf-8'),name,'exec')
    (SOURCE_DIR/name).write_bytes(data)
print('Frozen GP02B source:',BASELINE_ZIP,'| sha256:',FROZEN_WORKER_SHA)
print('Approved GP02C controller:',CONTROLLER_ZIP)
print('Controller files verified:',len(APPROVED_SOURCE))

## 2. Restore original agent specification and installed dependency contract (CPU)

No historical repair is re-executed here. The source, prompts, and original starter inputs are reconstructed from the archived **successful development-run record**. The historical six GP02B run results remain untouched. Note that the model is Gemma 4 **31B IT QAT w4a16 CT**; this is not the unquantized checkpoint.

In [ ]:
from zipfile import ZipFile
import importlib, hashlib, json, dataclasses

PROMPT_ORDER=('system_custom_base.md','investigation_protocol.md','tools_code_graph.md',
              'tools_code_retrieval.md','tools_file_operations.md','tools_testing.md','tools_execution.md')
with ZipFile(BASELINE_ZIP) as z:
    SETTINGS_V24=json.loads(z.read('settings.json'))
    BASELINE_TRIAL_CONFIG=json.loads(z.read('trial_config.json'))
    ARCHIVED_SERVER_CONFIG=json.loads(z.read('server_config.json'))
    SOURCE_GP02B={Path(m).name:z.read(m).decode('utf-8') for m in z.namelist()
                  if m.startswith('run_sources/') and m.count('/')==1 and m.endswith('.py')}
    PROMPTS_V27={name:z.read('run_sources/submission/prompts/'+name).decode('utf-8')
                 for name in PROMPT_ORDER}
    ORIGINAL_SAMPLING=z.read('run_sources/submission/configs/google_original_sampling.yaml').decode('utf-8')
    source_members=[m for m in z.namelist() if m.startswith('run_sources/submission/') and not m.endswith('/')]

if hashlib.sha256(SOURCE_GP02B['smoke_worker.py'].encode()).hexdigest()!=FROZEN_WORKER_SHA:
    raise RuntimeError('Frozen worker did not survive archive extraction')
composed='\n\n'.join('# COMPONENT: '+n+'\n'+PROMPTS_V27[n].strip() for n in PROMPT_ORDER)+'\n'
PROMPT_SHA_V27=hashlib.sha256(composed.encode()).hexdigest()
if PROMPT_SHA_V27!=SETTINGS_V24['expected_composed_system_sha256']:
    raise RuntimeError('Archived actual GP02B v2.7 prompt no longer matches the historical run')
if SETTINGS_V24['model_name']!='gemma-4-31b-it-qat-w4a16-ct' or SETTINGS_V24['max_output_tokens']!=2048:
    raise RuntimeError('Archived checkpoint or prompt-generation limit changed')

AGENT_DIR=SESSION_ROOT/'frozen_starter_input'
AGENT_DIR.mkdir()
with ZipFile(BASELINE_ZIP) as z:
    for member in source_members:
        rel=Path(member[len('run_sources/submission/'):])
        if rel.is_absolute() or '..' in rel.parts:raise RuntimeError('Unsafe starter member')
        target=AGENT_DIR/rel
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(z.read(member))
    original_filenames={
        'google_original_agent.yaml':'agent.yaml',
        'google_original_eval_config.yaml':'eval_config.yaml',
        'prompts/google_original_system.md':'prompts/system.md',
        'configs/google_original_sampling.yaml':'configs/sampling.yaml'}
    for source,target in original_filenames.items():
        (AGENT_DIR/target).write_bytes(z.read('run_sources/submission/'+source))

COMPETITION=Path(SETTINGS_V24['competition_root'])
if not (COMPETITION/'tasks.jsonl').is_file():
    raise FileNotFoundError('Official Kaggle competition dataset is missing')
if hashlib.sha256((COMPETITION/'tasks.jsonl').read_bytes()).hexdigest()!=BASELINE_TRIAL_CONFIG['tasks_sha256']:
    raise RuntimeError('Competition task manifest differs from frozen historical evidence')
TASK_ROWS={}
with (COMPETITION/'tasks.jsonl').open(encoding='utf-8') as f:
    for line in f:
        row=json.loads(line)
        if row.get('instance_id')==TASK_ID:
            TASK_ROWS[TASK_ID]={k:row[k] for k in
                ('instance_id','repo','base_commit','problem_statement','created_at') if k in row}
if len(TASK_ROWS)!=1 or TASK_ROWS[TASK_ID]['repo']!='Textualize/rich':
    raise RuntimeError('The predeclared public rich_2943 development task is missing')
if TASK_ROWS[TASK_ID]['base_commit']!=SETTINGS_V24['task_expected_commit']:
    raise RuntimeError('The public task base-commit does not match the frozen GP02B archive')
SNAPSHOT_PINS={TASK_ID:BASELINE_TRIAL_CONFIG['snapshot_pins'][TASK_ID]}
SNAPSHOT=COMPETITION/'snapshots'/f'{TASK_ID}.tgz'
if not SNAPSHOT.is_file() or hashlib.sha256(SNAPSHOT.read_bytes()).hexdigest()!=SNAPSHOT_PINS[TASK_ID]:
    raise RuntimeError('Frozen development source snapshot changed')
print('Frozen agent, prompt, starter inputs and source snapshot: PASS')
print('MODEL:',SETTINGS_V24['model_name'],'| QAT checkpoint:',SETTINGS_V24['model_path'])
print('GP02B v2.7 prompt SHA256:',PROMPT_SHA_V27)

## 3. Reuse the official offline Kaggle environment (CPU)

The following dependency and serving blocks reproduce the frozen GP02B setup. They run only when the required packages are not already importable. They do not install new models or change Gemma prompt/model configuration.

In [ ]:
import importlib
_REQUIRED=('adk_submission','swegemma','google.adk')
_missing=[]
for _name in _REQUIRED:
    try:importlib.import_module(_name)
    except ImportError:_missing.append(_name)
if _missing:
    print('Installing official frozen offline wheelhouse because these packages are missing:',_missing)
    import glob
    import importlib
    import logging
    import os
    import shutil
    import subprocess
    import sys
    from pathlib import Path

    # Configure environment variables for offline vLLM serving and LiteLLM routing
    os.environ['TRANSFORMERS_NO_TF'] = '1'
    os.environ['TRANSFORMERS_NO_FLAX'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['VLLM_NO_USAGE_STATS'] = '1'
    os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
    os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
    os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
    os.environ['NCCL_DEBUG'] = 'WARN'
    os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
    os.environ['LITELLM_TELEMETRY'] = 'False'
    os.environ['LITELLM_LOG'] = 'WARNING'
    os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
    os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['OTEL_SDK_DISABLED'] = 'true'
    os.environ['OTEL_PYTHON_DISABLED'] = 'true'


    def configure_logging() -> None:
        """Silence verbose third-party loggers."""
        for name in (
            'LiteLLM',
            'LiteLLM Router',
            'LiteLLM Proxy',
            'litellm',
            'httpx',
            'httpcore',
            'openai',
            'urllib3',
            'opentelemetry',
            'opentelemetry.context',
        ):
            lg = logging.getLogger(name)
            lg.setLevel(logging.WARNING)
            if 'litellm' in name.lower():
                lg.propagate = False
                lg.handlers.clear()


    configure_logging()

    WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

    # Remove broken cutlass .pth hooks if present
    for pth_pattern in (
        '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
        '/usr/local/lib/python*/site-packages/*cutlass*.pth',
    ):
        for pth in glob.glob(pth_pattern):
            try:
                os.unlink(pth)
            except OSError:
                pass

    # Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
    tmp_whl = Path('/tmp/wheelhouse')
    tmp_whl.mkdir(parents=True, exist_ok=True)
    for w in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in w.name.lower():
            continue
        target_name = (
            w.name.replace('cu128', '+cu128')
            if ('cu128' in w.name and '+' not in w.name)
            else w.name
        )
        target = tmp_whl / target_name
        if not target.exists():
            try:
                os.symlink(w, target)
            except OSError:
                shutil.copyfile(w, target)

    wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
    print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
        check=True,
    )
    importlib.invalidate_caches()
    configure_logging()
    print('Wheelhouse installation complete.')

    # CPU 05: verify installed distributions are importable in this exact kernel.
    # This is the mandatory gate before executing GPU model-startup code.
    import importlib
    _required_imports = {
        'adk_submission': 'adk-submission',
        'swegemma': 'swegemma',
        'google.adk': 'google-adk',
    }
    _failed_imports = {}
    for _module, _distribution in _required_imports.items():
        try:
            _pkg = importlib.import_module(_module)
            print(f'[PASS] {_module} (from {_pkg.__file__ if hasattr(_pkg, "__file__") else "namespace"})', flush=True)
        except Exception as _exc:
            _failed_imports[_module] = f'{type(_exc).__name__}: {_exc}'
            print(f'[FAIL] {_module}: {_failed_imports[_module]}', flush=True)
    if _failed_imports:
        raise RuntimeError(
            'CPU 05 offline package installation did not establish a usable runtime: '
            + repr(_failed_imports)
            + '. Verify that the attached wheelhouse contains adk-submission and its dependencies, '
            + 'and that pip uses the notebook kernel interpreter (sys.executable).'
        )
    print('CPU OFFLINE DEPENDENCY IMPORT GATE: PASS', flush=True)
else:
    print('All frozen official ADK/SWE-Gemma dependencies are importable; no installation needed')
for name in _REQUIRED:importlib.import_module(name)
print('Frozen dependency import gate: PASS')

## 4. Start the original Gemma 4 31B IT QAT vLLM service (L4 × 4)

This is **actual GPU code**, not a placeholder or CPU demonstration. It retains tensor parallelism 4, context 32,768, max 2,048 output tokens and the original QAT checkpoint. Startup can consume GPU quota; do not rerun this cell merely to save the notebook.

In [ ]:
# Start a NEW model service in this notebook/kernel when no service is already running.
# A restarted Kaggle session cannot reuse the previous Python server_instance.
import os, re, time, threading
from datetime import datetime, timezone
import requests, torch, litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')
MODEL_NAME = SETTINGS_V24['model_name']
MODEL_PATH = Path(SETTINGS_V24['model_path'])
assert MODEL_PATH.is_dir(), f'Attach original 31B model checkpoint: {MODEL_PATH}'
assert discover_declared_models(AGENT_DIR) == {MODEL_NAME} or MODEL_NAME in discover_declared_models(AGENT_DIR), 'Archived submission model differs from original.'
assert torch.cuda.is_available() and torch.cuda.device_count() == 4, 'Baseline requires four L4 GPUs (TP=4).'
assert all('L4' in torch.cuda.get_device_name(i) for i in range(4)), 'Expected original L4 x4 hardware.'
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters,'adapters',None) or {}
assert len(adapter_map) == 2, f'Unexpected v2.4 adapters: {list(adapter_map)}'
_v24_server = ARCHIVED_SERVER_CONFIG['config']
assert _v24_server['tensor_parallel_size'] == 4 and _v24_server['max_model_len'] == 32768

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16', gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True, enable_lora=True,
    max_loras=2, max_lora_rank=_v24_server['max_lora_rank'],
    scheduler_reserve_full_isl=False, tensor_parallel_size=4,
    startup_timeout=1200, extra_args=tuple(_v24_server['extra_args']),
)
# Avoid duplicate starts if the same kernel already owns a running server.
if globals().get('server_instance') is None:
    server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
    _stop = threading.Event()
    _started = time.monotonic()
    def _heartbeat():
        while not _stop.wait(10):
            print(f'[vLLM] loading {MODEL_NAME} | elapsed={time.monotonic()-_started:.0f}s',flush=True)
    _thread = threading.Thread(target=_heartbeat,daemon=True)
    _thread.start()
    try:
        print('Starting fresh Gemma 31B vLLM service on L4 x4...',flush=True)
        server_instance.start()
    finally:
        _stop.set()
        _thread.join(timeout=2)
    print(f'Server READY in {time.monotonic()-_started:.1f}s at {server_instance.base_url}',flush=True)
else:
    print('Reusing server_instance from THIS GP02B kernel:',server_instance.base_url)
response = requests.get(server_instance.base_url.rstrip('/')+'/models',
                        headers={'Authorization':'Bearer '+INFERENCE_API_KEY},timeout=20)
response.raise_for_status()
assert response.json().get('data'), 'Server has no registered models.'
models = server_instance.create_model_registry(
    aliases=[MODEL_NAME], model_prefix='openai/',api_key=INFERENCE_API_KEY)
SETTINGS_V24['starter_submission_dir'] = str(AGENT_DIR)
SETTINGS_V24['model_path'] = str(MODEL_PATH)
print('GP02B SERVING GATE: PASS | original model, adapters and sampling profile')

## 5. Install the experimental controller as readable source (CPU)

The code below is plain Python, not compressed/base64 notebook data. It inserts the GP02C controller **after** the verified original GP02B `context.create_tools` policy wrapper. It does not add mandatory investigation calls. Source identity and actual workspace correspondence are checked again inside the worker when the ADK sandbox has been created.

In [ ]:
%%writefile /kaggle/working/gp02c04b_live.py
"""GP02C-04B v1.1: live ADK experimental adapter for a *separate* frozen worker copy.

No change to the original GP02B v2.7 agent prompt, tool names or autonomy.
The controller is attached lazily after the official sandbox has been created.
The initial manifest is frozen from *agent-visible* repository tests only.
Protected verifier inputs and results are not available to this module.
"""
from __future__ import annotations
import asyncio
import functools
import hashlib
import inspect
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time
import traceback

_EXPECTED_WORKER_SHA = 'dbbdf4682dccb6d1b44a4d4dca3bb90a89b5b9bbe9070d4c36c16ec31e3eebfb'
_REQUIRED_ARM = ('B', 'C')
_CURRENT = None

class LiveIntegrationHold(RuntimeError):
    pass


def build_worker(frozen: str) -> str:
    """Derive a new worker only from the byte-exact observed six-run baseline."""
    actual=hashlib.sha256(frozen.encode('utf-8')).hexdigest()
    if actual != _EXPECTED_WORKER_SHA:
        raise LiveIntegrationHold('Frozen worker hash mismatch: '+actual)
    source_anchor='        context.create_tools = create_gated_tools\n'
    if frozen.count(source_anchor)!=1:
        raise LiveIntegrationHold('Original GP02B observational tool-policy boundary has changed')
    insertion=(source_anchor+
        '        # GP02C-04B: attach a separate, verified observation layer after GP02B policy.\n'
        '        from gp02c04b_live import install_lazy_controller as _install_gp02c\n'
        '        _install_gp02c(context=context, manager=manager, config=config, artifacts=a)\n')
    patched=frozen.replace(source_anchor,insertion)
    close_anchor='        if manager is not None:\n            manager.close()\n'
    if patched.count(close_anchor)!=1:
        raise LiveIntegrationHold('Original worker sandbox teardown boundary has changed')
    patched=patched.replace(close_anchor,
        '        # Seal GP02C evidence while the live sandbox and workspace still exist.\n'
        '        from gp02c04b_live import finish_live_worker as _finish_gp02c\n'
        '        _finish_gp02c(locals())\n'+close_anchor)
    # Original LiteLlm monitor is installed immediately before the official runner.
    observer_anchor='        restore_model_hook = install_model_observer(visible_event, a, config)\n'
    if patched.count(observer_anchor)!=1:
        raise LiveIntegrationHold('Original v2.7 model observer position has changed')
    patched=patched.replace(observer_anchor, observer_anchor+
        '        from gp02c04b_live import install_live_model_budget as _install_gp02c_model\n'
        '        _restore_gp02c_model_budget = _install_gp02c_model_budget(context)\n')
    model_restore_anchor='            restore_model_hook()\n            runner_module.build_agent_prompt = original_build_prompt\n'
    if patched.count(model_restore_anchor)!=1:
        raise LiveIntegrationHold('Original v2.7 model monitor restoration boundary has changed')
    patched=patched.replace(model_restore_anchor,
        '            try:\n'
        '                _restore_gp02c_model_budget()\n'
        '            finally:\n'
        '                restore_model_hook()\n'
        '                runner_module.build_agent_prompt = original_build_prompt\n')
    compile(patched,'gp02c04b_experimental_smoke_worker.py','exec')
    if patched.count('_install_gp02c(')!=1 or patched.count('_finish_gp02c(')!=1:
        raise LiveIntegrationHold('Experimental insertion cardinality mismatch')
    return patched


def _actual_workspace(context, manager) -> Path:
    cid=getattr(context,'container_id',None)
    sandboxes=getattr(manager,'sandboxes',None)
    if not cid or sandboxes is None:
        raise LiveIntegrationHold('Official sandbox/container identity not yet available; stop before inference')
    try:
        workspace=Path(sandboxes[cid]['workspace']).resolve(strict=True)
    except Exception as exc:
        raise LiveIntegrationHold('Cannot resolve the official sandbox host workspace') from exc
    if not workspace.is_dir() or not (workspace/'.git').is_dir():
        raise LiveIntegrationHold('Live workspace does not contain a Git source repository')
    return workspace


def _head(workspace):
    p=subprocess.run(['git','-C',str(workspace),'rev-parse','HEAD'],
        capture_output=True,text=True,timeout=15)
    if p.returncode or not re.fullmatch(r'[0-9a-f]{40}',p.stdout.strip()):
        raise LiveIntegrationHold('Cannot attest live base commit')
    return p.stdout.strip()


def _original_agent_view_probe(tools, head):
    """Read-only cross-view probe through the actual registered agent tool."""
    if not callable(tools.get('run_command')):
        raise LiveIntegrationHold('Original run_command not registered')
    result=tools['run_command'](command='cd /workspace && git rev-parse HEAD')
    if isinstance(result,str):
        try:result=json.loads(result)
        except (TypeError,ValueError) as exc:
            raise LiveIntegrationHold('Agent-side probe returned non-JSON text') from exc
    if not isinstance(result,dict):
        raise LiveIntegrationHold('Agent-side probe returned incompatible data')
    values=[result.get('stdout'),result.get('output')]
    nested=result.get('result')
    if isinstance(nested,dict): values.extend([nested.get('stdout'),nested.get('output')])
    if head not in [v.strip() for v in values if isinstance(v,str)]:
        raise LiveIntegrationHold('Host workspace and agent-visible source disagree')
    return {'head':head, 'probe':'PASS'}


def _read_check(workspace:Path, specification:dict):
    from gp02c_ledger import VisibleCheck, canonical_hash, definition_files_fingerprint
    from gp02c_live_adapter import observed_environment_identity
    if specification != {'check_id':'rich_style_visible','selector':'tests/test_style.py',
                         'expected_collected':26,'agent_aliases':['pytest tests/test_style.py',
                                                                  'python -m pytest tests/test_style.py']}:
        raise LiveIntegrationHold('Unapproved visible-test selection')
    selection=specification['selector']
    if not (workspace/selection).is_file():
        raise LiveIntegrationHold('Predeclared visible test is absent')
    definitions=(selection,)
    for candidate in ('tests/pytest.ini','pytest.ini','pyproject.toml','setup.cfg'):
        if (workspace/candidate).is_file():
            definitions+=(candidate,)
    env_id,_=observed_environment_identity()
    definition_hash=definition_files_fingerprint(workspace,definitions)
    argv=(sys.executable,'-m','pytest',selection)
    check=VisibleCheck(check_id=specification['check_id'],command=argv,
         definition_sha256=canonical_hash({'selection':selection,
                'argv':list(argv),'definition_files':list(definitions),
                'definition_files_sha256':definition_hash}),
         reserve_seconds=15.0,working_dir='.',test_selection=selection,
         expected_collected=specification['expected_collected'],
         pinned_environment_sha256=env_id,
         definition_files=definitions,definition_files_sha256=definition_hash)
    return check


def install_lazy_controller(*,context,manager,config,artifacts):
    """Attach only when official tool registry is materialized in live sandbox."""
    global _CURRENT
    if _CURRENT is not None:
        raise LiveIntegrationHold('Controller already active in this worker')
    arm=os.environ.get('GP02C04B_ARM')
    task=config.get('task_id')
    seed=int(os.environ.get('GP02C04B_SEED','42'))
    run_id=os.environ.get('GP02C04B_RUN_ID')
    root=os.environ.get('GP02C04B_EVIDENCE_ROOT')
    if arm not in _REQUIRED_ARM or task!='rich_2943' or not run_id or not root:
        raise LiveIntegrationHold('Only the registered rich_2943 B/C development pilot is allowed')
    source_sha=os.environ.get('GP02C04B_EXPECTED_WORKER_SHA')
    if source_sha != _EXPECTED_WORKER_SHA:
        raise LiveIntegrationHold('Frozen worker source was not authenticated')
    if config['max_tool_calls']!=60 or config['max_llm_calls']!=60 or float(config['agent_time_minutes'])!=15:
        raise LiveIntegrationHold('Original agent budgets were changed')
    if config['model_name']!='gemma-4-31b-it-qat-w4a16-ct' or config['max_output_tokens']!=2048:
        raise LiveIntegrationHold('Original quantized Gemma model or generation cap changed')
    orig=context.create_tools
    state={'arm':arm,'task_id':task,'seed':seed,'run_id':run_id,
           'evidence_root':Path(root).resolve(), 'artifacts':Path(artifacts),
           'original_create_tools':orig,'wiring':None,'evidence':None,
           'started':time.monotonic(),'configuration':config,'finished':False}
    _CURRENT=state

    def lazy_tools():
        # Registered tool mappings are built by the existing GP02B observational policy.
        if state['wiring'] is not None:
            return orig()
        unwrapped=orig()
        if not isinstance(unwrapped,dict):
            raise LiveIntegrationHold('Unexpected actual ADK tool registry type')
        host=_actual_workspace(context,manager)
        expect=config.get('task_expected_commit')
        if _head(host)!=expect:
            raise LiveIntegrationHold('Official sandbox base commit mismatches public task')
        probe=_original_agent_view_probe(unwrapped,expect)
        from gp02c_ledger import VisibleCheck
        from gp02c04_pilot_wiring import attach_dev_pilot
        from gp02c_evidence_export import RunEvidence
        check=_read_check(host,{'check_id':'rich_style_visible','selector':'tests/test_style.py',
                       'expected_collected':26,
                       'agent_aliases':['pytest tests/test_style.py','python -m pytest tests/test_style.py']})
        # The study requires identical B/C manifest; lock public source/check inputs,
        # and compare this immutable manifest digest for the second arm.
        manifest={
            'task_id':task,'base_commit':expect,'seed':seed,
            'checks':[{**check.__dict__,'command':list(check.command),
                       'definition_files':list(check.definition_files)}],
            'aliases':{'pytest tests/test_style.py':check.check_id,
                       'python -m pytest tests/test_style.py':check.check_id},
            'source_head':_head(host), 'source_view_probe':probe,
        }
        lock=state['evidence_root'].parent/'GP02C04B_visible_manifest_lock.json'
        lock.parent.mkdir(parents=True,exist_ok=True)
        new_bytes=(json.dumps(manifest,sort_keys=True,indent=2)+'\n').encode()
        if lock.is_file():
            if lock.read_bytes()!=new_bytes:
                raise LiveIntegrationHold('B/C frozen visible-test manifest mismatch')
        else:
            with lock.open('xb') as stream:stream.write(new_bytes)
        base=state['evidence_root']/(run_id+'_baseline_source')
        if base.exists():raise LiveIntegrationHold('Per-attempt baseline already exists')
        shutil.copytree(host,base,symlinks=False)
        rec=RunEvidence(state['evidence_root'],run_id=run_id,task_id=task,arm=arm,
                         seed=seed,attempt_index=1,
                         setup={'checkpoint':'gemma-4-31b-it-qat-w4a16-ct',
                         'frozen_worker_sha256':_EXPECTED_WORKER_SHA,
                         'public_task':task,'base_commit':expect,
                         'visible_manifest_sha256':hashlib.sha256(new_bytes).hexdigest(),
                         'source_view_probe':probe,'gate':'LIVE_UNGRADED'})
        rec.event('ACTUAL_ADK_REGISTRY_ACCEPTED',{'names':sorted(unwrapped),'host_head':expect})
        state['evidence']=rec
        state['source_root']=host
        state['baseline_source']=base
        state['manifest_hash']=hashlib.sha256(new_bytes).hexdigest()
        state['wiring']=attach_dev_pilot(context=context,task_id=task,arm=arm,
                            run_id=run_id,seed=seed,
                            sandbox_host_workspace=host,baseline_source=base,
                            evidence_directory=state['evidence_root']/(run_id+'_trusted_checks'),
                            checks=(check,),visible_command_aliases=manifest['aliases'],
                            started_monotonic=state['started'],
                            run_evidence=rec)
        # Call the newly installed outer GP02C wrapper. Its inner lazy hook sees wiring.
        return context.create_tools()

    @functools.wraps(orig)
    def delegated():return lazy_tools()
    context.create_tools=delegated
    return state


def install_live_model_budget(context):
    """Wrap AFTER the original GP02B model monitor; no changes to generation parameters."""
    from google.adk.models.lite_llm import LiteLlm
    original=LiteLlm.generate_content_async
    if not callable(original):raise LiveIntegrationHold('ADK LiteLlm model API missing')
    def precheck():
        if _CURRENT is None or _CURRENT['wiring'] is None:
            raise LiveIntegrationHold('ADK tool controller was not attached before first model request')
        ctl=_CURRENT['wiring']
        budget=ctl.controller.budget
        if budget.used_model_calls>=budget.max_model_calls:
            raise LiveIntegrationHold('Shared model-call budget exhausted')
        remaining=ctl.deadline.remaining('ordinary')
        if remaining<=0:raise TimeoutError('840-second repair model deadline reached')
        ctl.bridge.sync_model_calls(budget.used_model_calls+1)
        return remaining
    if inspect.isasyncgenfunction(original):
        @functools.wraps(original)
        async def intercept(*a,**kw):
            seconds=precheck()
            async with asyncio.timeout(seconds):
                async for e in original(*a,**kw):yield e
    elif inspect.iscoroutinefunction(original):
        @functools.wraps(original)
        async def intercept(*a,**kw):
            seconds=precheck()
            return await asyncio.wait_for(original(*a,**kw),seconds)
    else:raise LiveIntegrationHold('Model method does not expose supported async shape')
    LiteLlm.generate_content_async=intercept
    def restore():LiteLlm.generate_content_async=original
    return restore


def _production_patch(workspace:Path, evidence_root:Path):
    """Capture eligible tracked and untracked source edits, excluding tests/scratch.

    The frozen production-only rule is used identically for B and C. The original
    runner's agent.patch is independently retained for historical comparability.
    """
    def eligible(rel):
        parts=Path(rel).parts
        if not parts or not rel.endswith('.py') or any(p.lower() in {
           'test','tests','example','examples','scratch','scripts','__pycache__'} for p in parts):
            return False
        return not (Path(rel).name.startswith('test_') or Path(rel).name.endswith('_test.py'))
    changed=subprocess.run(['git','-C',str(workspace),'diff','--name-only','-z','HEAD','--'],
            capture_output=True,timeout=20)
    unknown=subprocess.run(['git','-C',str(workspace),'ls-files','--others','--exclude-standard','-z'],
            capture_output=True,timeout=20)
    if changed.returncode or unknown.returncode:
        raise LiveIntegrationHold('Cannot enumerate source changes')
    edited=[p.decode('utf-8','surrogateescape') for p in changed.stdout.split(b'\0') if p]
    untracked=[p.decode('utf-8','surrogateescape') for p in unknown.stdout.split(b'\0') if p]
    tracked=[p for p in edited if eligible(p)]
    fresh=[p for p in untracked if eligible(p)]
    combined=b''
    if tracked:
        diff=subprocess.run(['git','-C',str(workspace),'diff','--binary','HEAD','--',*tracked],
                     capture_output=True,timeout=25)
        if diff.returncode:
            raise LiveIntegrationHold('Could not capture tracked source patch')
        combined+=diff.stdout
    for rel in fresh:
        full=workspace/rel
        if not full.is_file() or full.is_symlink():
            raise LiveIntegrationHold('Untracked candidate source is not a regular file: '+rel)
        patch=subprocess.run(['git','diff','--no-index','--binary','--','/dev/null',rel],
                      cwd=str(workspace),capture_output=True,timeout=25)
        if patch.returncode != 1 or b'new file mode' not in patch.stdout:
            raise LiveIntegrationHold('Could not capture new production source: '+rel)
        combined+=patch.stdout
    candidate_dir=evidence_root/'agent'
    candidate_dir.mkdir(parents=True,exist_ok=True)
    patch_path=candidate_dir/'production_candidate.patch'
    patch_path.write_bytes(combined)
    manifest={'tracked':tracked,'untracked':fresh,
              'excluded_changed_files':sorted(set(edited+untracked)-set(tracked+fresh)),
              'production_patch_sha256':hashlib.sha256(combined).hexdigest(),
              'production_patch_bytes':len(combined)}
    (candidate_dir/'production_extraction_manifest.json').write_text(
        json.dumps(manifest,indent=2)+'\n')
    return patch_path, manifest


def finish_live_worker(locals_dict):
    """Seal evidence before original worker closes/deletes the sandbox.
    Never claim protected grading from this stage.
    """
    state=_CURRENT
    if state is None or state.get('finished'):return
    state['finished']=True
    wiring=state.get('wiring')
    rec=state.get('evidence')
    if wiring is None or rec is None:
        # No model inference must succeed without tool integration; the worker
        # exception is retained in its original artifact log.
        return
    try:
        try:
            path,meta=_production_patch(state['source_root'],rec.dir)
        except Exception as exc:
            # The official ADK runner may already have closed its sandbox when
            # it returns. In that case preserve the native patch rather than
            # losing the attempt's complete event archive.
            original=Path(locals_dict.get('a',state['artifacts']))/'agent/agent.patch'
            if not original.is_file():
                raise
            path=original
            meta={'fallback':'ORIGINAL_RUNNER_PATCH','workspace_capture_error':
                  type(exc).__name__+': '+str(exc)[:400],
                  'trusted_source_workspace_recheck':'NOT_AVAILABLE'}
        rec.event('PRODUCTION_PATCH_CAPTURE',meta)
        # The complete, allowlisted original GP02B *agent-visible* trajectory is
        # archived separately from the protected independent evaluator.
        original_agent_dir=Path(locals_dict.get('a',state['artifacts']))/'agent'
        for filename in ('events.jsonl','events_full.jsonl','agent.patch',
                         'sandbox_exec.jsonl','model_responses.json',
                         'tool_diagnostics.json','initial_task_prompt.txt',
                         'investigation_gate_state.json'):
            source=original_agent_dir/filename
            if source.is_file():
                rec.capture('agent/original_gp02b/'+filename,source)
        transport_calls=original_agent_dir/'model_transport/calls.jsonl'
        if transport_calls.is_file():
            rec.capture('agent/original_gp02b/model_transport_calls.jsonl',transport_calls)
        submitted=wiring.bridge.submission_status in ('SUBMITTED_GATE_VALID',)
        status='SUBMITTED_GATE_VALID' if submitted else wiring.bridge.submission_status
        wiring.record_run_evidence(termination='WORKER_EXITED',delivery=status,
                      patch_file=path,protected_grade='NOT_RUN')
    except BaseException as exc:
        try:rec.event('FINALIZATION_HOLD',{'type':type(exc).__name__,
                       'message':str(exc)[:1000]})
        except Exception:pass
        raise
    finally:
        wiring.close()


In [ ]:
# Copy the exact, already displayed plain-text adapter into this session's
# independent working directory. Do not rely on /kaggle/working across sessions.
shutil.copyfile('/kaggle/working/gp02c04b_live.py',SOURCE_DIR/'gp02c04b_live.py')
if str(SOURCE_DIR) not in sys.path:sys.path.insert(0,str(SOURCE_DIR))
from gp02c04b_live import build_worker
EXPERIMENTAL_WORKER=build_worker(SOURCE_GP02B['smoke_worker.py'])
EXPERIMENTAL_SOURCES={**SOURCE_GP02B,'smoke_worker.py':EXPERIMENTAL_WORKER}
for name,src in SOURCE_GP02B.items():
    if name!='smoke_worker.py' and EXPERIMENTAL_SOURCES[name]!=src:
        raise RuntimeError('An unrelated original GP02B source was changed')
(SOURCE_DIR/'frozen_smoke_worker.py').write_text(SOURCE_GP02B['smoke_worker.py'])
(SOURCE_DIR/'experimental_smoke_worker.py').write_text(EXPERIMENTAL_WORKER)
print('EXACT BASELINE SHA:',FROZEN_WORKER_SHA)
print('Experimental derivative SHA:',hashlib.sha256(EXPERIMENTAL_WORKER.encode()).hexdigest())
print('Agent prompt, tool names, original policy and model selection unchanged')

## 6. Construct the actual isolated development trials (CPU preparation)

This invokes only the original GP02B `Trial` setup and qualification stages on the **predeclared development task**. We do not rerun the old baseline results. Each experimental arm gets a fresh source/sandbox. Protected qualification is researcher-only and never appears in the public task supplied to Gemma. **The 36-task held-out cohort is not opened or graded.**

In [ ]:
import importlib.util
from pathlib import Path
CORE_PINS_V24=[
 {'distribution':'adk-eval-core','version':'0.1.0','sha256':'e614e1b2fe8d886a634871e4c57f9887e1cc66e8c27300f8503a43c6e3ca82f3'},
 {'distribution':'adk-submission','version':'0.2.13','sha256':'f156f0bd84580ccb9334726d114d7a5481f16d3a6e2887cc4d276ced0ef8d99d'},
 {'distribution':'swegemma','version':'0.2.11','sha256':'9ad876581f3acde8ec3c369858f0834e2b9f41a3cffce3ca42f09cb090aea9d3'},
]
BASE_ASSETS={
    'tasks_sha256':BASELINE_TRIAL_CONFIG['tasks_sha256'],
    'snapshot_sha256':SNAPSHOT_PINS[TASK_ID],
    'verification_source_sha256':BASELINE_TRIAL_CONFIG['verification_source_sha256'],
    'agent_runner_source_sha256':BASELINE_TRIAL_CONFIG['agent_runner_source_sha256'],
    'core_pins':CORE_PINS_V24,
    'sources':EXPERIMENTAL_SOURCES,
    'protocols':PROMPTS_V27,
}
workflow_file=SOURCE_DIR/'workflow.py'
workflow_file.write_text(SOURCE_GP02B['workflow.py'])
spec=importlib.util.spec_from_file_location('gp02c04b_frozen_workflow',workflow_file)
workflow_gp02b=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=workflow_gp02b
spec.loader.exec_module(workflow_gp02b)

from gp02c_worker_supervisor import supervise_worker
from gp02c_evidence_export import RunEvidence

def build_trial(arm):
    settings=copy.deepcopy(SETTINGS_V24)
    settings.update(task_id=TASK_ID,task_expected_repo=TASK_ROWS[TASK_ID]['repo'],
        task_expected_commit=TASK_ROWS[TASK_ID]['base_commit'],
        output_base=str(PILOT_ROOT/arm),
        starter_server={'config':dataclasses.asdict(vllm_cfg),
                        'base_url':server_instance.base_url,
                        'command':server_instance.build_cmd(),
                        'log_path':server_instance.log_path,
                        'lifecycle_owner':'official_starter'},
        starter_submission_dir=str(AGENT_DIR),
        expected_composed_system_sha256=PROMPT_SHA_V27)
    trial=workflow_gp02b.Trial(settings,copy.deepcopy(BASE_ASSETS))
    for path in SOURCE_DIR.glob('gp02c*.py'):
        shutil.copy2(path,trial.root/path.name)
    return trial

print('Live Trial type:',workflow_gp02b.Trial)
print('Fresh workspaces will be created for B/C only. No model repair yet.')

## 7. LIVE Gemma 4 31B development integration — B then C (L4 × 4)

**This cell launches the real ADK model worker.** Unlike the earlier CPU-only 04B preflight, it can consume L4 quota and produces actual model/tool trajectories.

Before each inference, the worker must verify its actual ADK tool registry, the host/agent workspace identity, the pinned visible `tests/test_style.py` check and the original v2.7 observational policy. The originally observed development check collects 26 tests; a mismatch is an **integration hold**, not a passing result. Rechecks and the final gate are charged against the existing 900-second and 60-call budgets. The original vLLM server remains outside the worker's subprocess group.

The first B attempt is preserved even if it fails. **C executes only if B reached a verified live ADK attachment and generated its own evidence ZIP.** Do not rerun a failed model attempt merely to improve its outcome. Independent evaluation is invoked afterwards and recorded separately.

In [ ]:
import time, os, hashlib, zipfile, json
LIVE_RESULTS={}
TRIALS={}
LIVE_RUN_REGISTER=SESSION_ROOT/'GP02C04B_Live_Register.json'
if LIVE_RUN_REGISTER.exists():
    raise RuntimeError('Refusing to reuse an existing GP02C-04B attempt register')
LIVE_RUN_REGISTER.write_text('{}\n')

for arm in ARMS:
    # A fresh task, agent context and sandbox is constructed for each arm.
    trial=build_trial(arm)
    TRIALS[arm]=trial
    for stage,action in [(2,trial.discover),(3,trial.install),(4,trial.prepare),
                         (5,trial.qualify),('5B',trial.tool_preflight)]:
        trial.step(stage,action)
        if trial.holds or (stage==5 and not trial.qualified) or (stage=='5B' and not trial.tools_ready):
            print('PRE-INFERENCE INTEGRATION HOLD',arm,trial.holds[-1] if trial.holds else '')
            break
    if trial.holds or not trial.qualified or not trial.tools_ready:
        LIVE_RESULTS[arm]={'status':'PRE_INFERENCE_HOLD','holds':trial.holds,'trial_root':str(trial.root)}
        break

    # Fresh, no-overwrite identity. The worker will append evidence while running.
    attempt_id='GP02C04B_'+arm+'_'+RUN_TAG
    evidence_dir=EVIDENCE_ROOT/arm
    evidence_dir.mkdir(exist_ok=False)
    env=os.environ.copy()
    env.update({'GP02C04B_ARM':arm,'GP02C04B_SEED':str(SEED),
                'GP02C04B_RUN_ID':attempt_id,
                'GP02C04B_EVIDENCE_ROOT':str(evidence_dir),
                'GP02C04B_EXPECTED_WORKER_SHA':FROZEN_WORKER_SHA,
                'PIP_NO_INPUT':'1','LITELLM_TELEMETRY':'False','DO_NOT_TRACK':'1'})
    start=time.monotonic()
    command=(str(trial.python),'-u',str(trial.root/'smoke_worker.py'),
             '--config',str(trial.config_path))
    # Separate worker process group, physical 900-second limit; GPU service is
    # controlled by the notebook kernel, NEVER by this supervisor.
    result=supervise_worker(command,cwd=trial.root,evidence_root=evidence_dir,
                            overall_seconds=900.0,env=env)
    worker_status=trial.artifacts/'agent/result.json'
    trial.agent_result=json.loads(worker_status.read_text()) if worker_status.is_file() else {}
    trial.agent_result['worker_process']=result
    trial.save('agent/result.json',trial.agent_result)
    if result['timed_out'] and not (evidence_dir/(attempt_id+'.zip')).exists():
        try:
            recovery=RunEvidence.recover_interrupted(evidence_dir,attempt_id,
                           termination='TIMEOUT',delivery='UNVERIFIED_BUDGET')
            print('Recovered interrupted evidence:',recovery['zip_path'])
        except Exception as exc:
            print('Recovery not possible; preserving raw worker journal:',type(exc).__name__,str(exc))

    archive_path=evidence_dir/(attempt_id+'.zip')
    if archive_path.is_file():
        with zipfile.ZipFile(archive_path) as z:
            actual_live=('events.jsonl' in z.namelist() and
                         b'ACTUAL_ADK_REGISTRY_ACCEPTED' in z.read('events.jsonl'))
    else:actual_live=False
    row={'arm':arm,'task_id':TASK_ID,'seed':SEED,'attempt_id':attempt_id,
         'worker':result,'actual_live_adk_attached':actual_live,
         'agent_model_calls':trial.agent_result.get('agent_model_calls'),
         'worker_complete':trial.agent_result.get('worker_complete'),
         'agent_patch_bytes':trial.agent_result.get('patch_bytes'),
         'submitted':trial.agent_result.get('patch_submitted'),
         'evidence_zip':str(archive_path) if archive_path.is_file() else None,
         'trial_root':str(trial.root),'protected_grade':'NOT_RUN'}
    LIVE_RESULTS[arm]=row
    LIVE_RUN_REGISTER.write_text(json.dumps(LIVE_RESULTS,indent=2,default=str)+'\n')
    print('LIVE',arm,'| attached:',actual_live,
          '| worker exit:',result['returncode'],
          '| model calls:',row['agent_model_calls'],'| evidence:',row['evidence_zip'])
    if not actual_live or (row['agent_model_calls'] or 0)<1 or not row['worker_complete']:
        print('LIVE ADK INTEGRATION HOLD: worker attachment, model-call evidence, or completion is missing. No further arm will run.')
        break

print('Live B/C register:',LIVE_RUN_REGISTER)

## 8. Researcher-only independent verification and immutable sidecars

Protected verification must occur **after** the agent has finished and its GP02C ZIP is sealed. A protected grade is not sent back to the Gemma model or controller. This cell performs independent grading only for workers that completed a repair and left a suitable candidate patch; rejected submissions remain separately classified from independently graded correctness.

In [ ]:
import hashlib,json
from pathlib import Path
for arm,row in LIVE_RESULTS.items():
    if not row.get('actual_live_adk_attached'):continue
    trial=TRIALS[arm]
    trial_dir=trial.root
    config_file=trial.config_path
    source_patch=trial.artifacts/'agent/agent.patch'
    if not (config_file.is_file() and source_patch.is_file()):
        row['protected_grade']='NOT_RUN'
        row['grading_reason']='No independently gradeable frozen runner patch'
        continue
    # Use the unchanged GP02B independent evaluator only AFTER the agent ZIP is sealed.
    candidate_report=SESSION_ROOT/f'GP02C04B_{arm}_protected_grade_sidecar.json'
    if candidate_report.exists():raise RuntimeError('Immutable grade sidecar already exists')
    # Official Trial.verify() safely skips if the worker was incomplete.
    try:
        trial.step(7,trial.verify)
        grade=trial.candidate_result
        grade_value='NOT_RUN'
        if isinstance(grade,dict):
            from pathlib import Path
            import runpy
            classify=runpy.run_path(str(trial.root/'smoke_worker.py'),run_name='gp02c04b_grader_helpers')['classify_candidate']
            value=classify(grade)
            grade_value='PASS' if value=='PATCH_VERIFIED' else ('INFRA_ERROR' if value=='VERIFICATION_HOLD' else 'FAIL')
        elif trial.holds:
            grade_value='INFRA_ERROR'
            grade={'grading_hold':trial.holds}
    except Exception as exc:
        grade_value='INFRA_ERROR'
        grade={'grading_error':type(exc).__name__+': '+str(exc)}
    evidence=Path(row['evidence_zip'])
    sidecar={'schema':'gp02c04b.protected-sidecar.v1','task_id':TASK_ID,
        'arm':arm,'seed':SEED,'attempt_id':row['attempt_id'],
        'agent_evidence_zip_sha256':hashlib.sha256(evidence.read_bytes()).hexdigest(),
        'agent_patch_sha256':hashlib.sha256(source_patch.read_bytes()).hexdigest(),
        'protected_grade':grade_value,'protected_report':grade,
        'protected_results_never_agent_visible':True}
    with candidate_report.open('x',encoding='utf-8') as f:
        json.dump(sidecar,f,indent=2,default=str)
    row['protected_grade']=grade_value
    row['grade_sidecar']=str(candidate_report)
    print('INDEPENDENT GRADE',arm,grade_value)
LIVE_RUN_REGISTER.write_text(json.dumps(LIVE_RESULTS,indent=2,default=str)+'\n')
print('Independent grading sidecars (if produced) are separate from sealed agent evidence ZIPs.')

## 9. Output evidence and live acceptance decisions

All actual run archives are stored under `/kaggle/working/GP02C_04B_Live_<session>/attempt_evidence`. Keep original ZIPs immutable and preserve unsuccessful attempts. Use **Save Version → Quick Save** to preserve completed Kaggle outputs without rerunning GPU work; download the archives to your locally Google Drive-synced project folder.

In [ ]:
import zipfile,json,hashlib
STATUS={arm:{k:row.get(k) for k in
    ('actual_live_adk_attached','worker_complete','agent_model_calls','submitted',
     'agent_patch_bytes','protected_grade','evidence_zip')}
    for arm,row in LIVE_RESULTS.items()}
all_live=(len(STATUS)==len(ARMS) and all(r.get('actual_live_adk_attached') and
   (r.get('agent_model_calls') or 0)>0 and r.get('evidence_zip') for r in STATUS.values()))
conclusion='LIVE_B_C_DEVELOPMENT_ATTEMPTS_RECORDED' if all_live else 'LIVE_INTEGRATION_INCOMPLETE_OR_HOLD'
REPORT={'stage':'GP02C-04B v1.1','execution':'REAL_L4_MODEL_IF_RUN',
  'task':TASK_ID,'arms':list(ARMS),'frozen_worker_sha256':FROZEN_WORKER_SHA,
  'model':'gemma-4-31b-it-qat-w4a16-ct','results':STATUS,
  'conclusion':conclusion,'research_result_claim':'NONE_PRIOR_TO_INDEPENDENT_ANALYSIS'}
path=SESSION_ROOT/'GP02C04B_Live_Acceptance_Report.json'
with path.open('x') as f:json.dump(REPORT,f,indent=2)
print(json.dumps(REPORT,indent=2))
for arm,row in LIVE_RESULTS.items():
    if row.get('evidence_zip'):
        display(FileLink(row['evidence_zip']))
    if row.get('grade_sidecar'):
        display(FileLink(row['grade_sidecar']))
display(FileLink(str(path)))

### Researcher observations — complete after executing in Kaggle

Record the actual B/C source edits, first qualifying visible PASS, relevant subsequent edit, invalidated receipts, C proactive test executions, final-gate acceptance, independent grades, total model/tool calls and timeouts. 

**Do not report C as improving repair quality from this pilot.** The 36 held-out tasks remain sealed. If both B and C run but no naturally occurring PASS→edit interval is observed, the next step is the preregistered feasibility decision—not retroactive tuning of tasks based on hidden outcomes.